# PROPER MOTIONS AND VELOCITIES WITH EUCLID

In this notebook we want to show how important could be to point the Euclid space telescope towards the center of the Milky way galaxy (CG) in order to study the kinematics of its inhabitants (stars, not alien) by using simulated data of this region thanks to the Besancon model. 

To do so, we need to compute the errors on the velocities (V_phi, V_R) from the gaussian error given by Euclid on the proper motions (mux, muy). We should then have a coherent situation of what Euclid will see if pointing in that direction.

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from scipy.stats import norm

In [2]:
#data = [df_fits, 
        #all_J,
        #bulge_J  
        #RC_J_seen, 
        #RC_J_I_seen, 
        #RC_J_I_V_seen, 
        #pop_filter(RC_J_seen)[0], # resolved Red clump with J < 19 from the thin disk
        #pop_filter(RC_J_seen)[1], # resolved Red clump with J < 19 from the bulge
        #pop_filter(RC_J_I_seen)[0], # resolved Red clump with J < 19 and I < 21 from the thin disk
        #pop_filter(RC_J_I_seen)[1] # resolved Red clump with J < 19 and I < 21 from the bulge
        #RC
        #]
df_fits = joblib.load('data_0.joblib')
all_J = joblib.load('data_1.joblib')
bulge_J = joblib.load('data_2.joblib')
RC_J_seen = joblib.load('data_3.joblib')
RC_J_I_seen = joblib.load('data_4.joblib')
RC_J_thin_seen = joblib.load('data_6.joblib')
RC_J_bulge_seen = joblib.load('data_7.joblib')
RC_J_I_thin_seen = joblib.load('data_8.joblib')
RC_J_I_bulge_seen = joblib.load('data_9.joblib')
RC = joblib.load('data_10.joblib')

Let's remember that we are working with a simulation

 - centered on l = 1.0 deg, b = 0.0 deg
 - 10 kpc of distance
 - with 10 < J < 21 and 
 - f.o.v = 1 deg^2

Let's remember that in "Selection of RED CLUMP" file I put x_Gal = -x_Gal to make the Besancon coordinate system a left-handed reference frame.
Now let's go back to its original system, a right-handed.

In [3]:
df_fits["x_Gal"] = - df_fits["x_Gal"] 

Let's verify that we obtain the same results. 
First of all I want to verify that the coordinates are indeed the same, when passing from longitude (l) and latitude (b) to galactic coordinate (x_Gal, y_Gal and z_Gal). 
The formula used by Besancon model should be:
- x_Gal = dist * cos(l) * cos(b) - 8
- y_Gal = dist * sin(l) * cos(b)
- z_Gal = dist * sin(b)

In [4]:
difference_x = df_fits["x_Gal"] - df_fits["Dist"]*np.cos(np.radians(df_fits["longitude"])) * np.cos(np.radians(df_fits["latitude"])) + 8
difference_y = df_fits["y_Gal"] - df_fits["Dist"]*np.sin(np.radians(df_fits["longitude"])) * np.cos(np.radians(df_fits["latitude"]))
difference_z = df_fits["z_Gal"] - df_fits["Dist"]*np.sin(np.radians(df_fits["latitude"]))

#print(difference_y.head())
print(difference_x.describe(), difference_y.describe(), difference_z.describe())
#print("Max difference in x_Gal:", np.max(np.abs(difference_x)))
#print("Max difference in y_Gal:", np.max(np.abs(difference_y)))
#print("Max difference in z_Gal:", np.max(np.abs(difference_z))) 

count    2.681096e+06
mean    -3.739664e-07
std      4.083278e-05
min     -1.006563e-04
25%     -2.967819e-05
50%     -3.548993e-07
75%      2.892609e-05
max      9.988293e-05
dtype: float64 count    2.681096e+06
mean    -2.581871e-08
std      2.887592e-05
min     -5.122715e-05
25%     -2.502696e-05
50%     -5.587196e-08
75%      2.497346e-05
max      5.127942e-05
dtype: float64 count    2.681096e+06
mean     1.500000e-02
std      2.886729e-05
min      1.494956e-02
25%      1.497501e-02
50%      1.500001e-02
75%      1.502499e-02
max      1.505045e-02
dtype: float64


The difference is really small hence they are correct (it's not exactly 0 because there are some approximation error maybe).

Now let's do the same, between the cartesian galactic velocities w.r.t. the LSR given (UU, VV and WW) and the proper motion mux and muy (mu_l and mu_b). 
We will need also the radial velocity HRV. We will need also to adjust the velocities taking into account the relative motion of the sun.

The main reason of these verifications is to test consistencies of the Besancon model.

In [27]:
def galactic_uvw(mu_l, mu_b, l_deg, b_deg, d_kpc, Vr):
    # Convert proper motions from mas/yr to arcsec/yr:
    mu_l_arcsec = mu_l / 1000.0
    mu_b_arcsec = mu_b / 1000.0

    # Convert distance from kpc to pc
    d_pc = d_kpc * 1000.0 

    # Convert angles from degrees to radians:
    l = np.radians(l_deg)
    b = np.radians(b_deg)
    
    # Conversion factor from (arcsec/yr * pc) to km/s:
    k = 4.74047  # km/s 
    
    # Compute tangential velocities along l and b:
    V_l = k * d_pc * mu_l_arcsec
    V_b = k * d_pc * mu_b_arcsec
    
    # Compute U, V, W (cartesian velocities wrt the GC):
    U = Vr * np.cos(l) * np.cos(b) - (1/k) * (V_l * np.cos(b) * np.sin(l) + V_b * np.sin(b)*np.cos(l)) - 12.8 
    V = Vr * np.sin(l) * np.cos(b) + (1/k) * (V_l * np.cos(l) * np.cos(b) - V_b * np.sin(l) * np.sin(b)) - 244.6 - 0.9
    W = Vr * np.sin(b) + (1/k) * V_b * np.cos(b) -7.1
    
    return U, V, W

In [28]:
Difference_U = df_fits["UU"] - galactic_uvw(df_fits["mux"], df_fits["muy"], df_fits["longitude"], df_fits["latitude"], df_fits["Dist"], df_fits["HRV"])[0]
print(np.abs(Difference_U).mean())

Difference_V = df_fits["VV"] - galactic_uvw(df_fits["mux"], df_fits["muy"], df_fits["longitude"], df_fits["latitude"], df_fits["Dist"], df_fits["HRV"])[1]
print(np.abs(Difference_V).mean())

Difference_W = df_fits["WW"] - galactic_uvw(df_fits["mux"], df_fits["muy"], df_fits["longitude"], df_fits["latitude"], df_fits["Dist"], df_fits["HRV"])[2]
print(np.abs(Difference_W).mean())

119.53989433844252
172.38554700219507
42.34514421606768


The difference is way too large so something must be wrong.